## PTM Annotations with Sitetack

Lead     : `<Claire / clairecyuan>`

Issue    : [Github Issue #91](https://github.com/petadex/igem-toronto/issues/91) — _PTM Annotation Sitetack_

Start    : `2026-07-06`

Complete : `2026-08-01`

Files    : `~igem-toronto/resources/sitetack_template`

S3 files : `s3://sitetack/prediction_results_batch_10000.csv` pred results for 10,000 sequences from petadex

---

### Data Accessed
```python
# For local work:
session = boto3.Session(profile_name='petadex-claire')
s3 = session.client('s3')
obj = s3.get_object(Bucket='petadex-orf-fastaa', Key='blastnr_pazy.catalytic_orfs.fa')
body = obj['Body']
records = SeqIO.parse('blastnr_pazy.catalytic_orfs.fa', 'fasta')
```
In EC2, the whole dataset was directly downloaded with:
```bash
aws s3 cp s3://petadex-orf-fastaa/blastnr_pazy.catalytic_orfs.fa
```

## Introduction



## Objectives

1. Test various models on ground truth dataset
2. Benchmark time and cost for running sequences on a small set
3. Run PTM predictions for all Petadex sequences

---

## Materials and Methods

### System Initialization
Clone the sitetack repo to your machine. 
```bash
git clone https://github.com/clair-gutierrez/sitetack
```
All the models used are stored in sitetack/sitetack/models.  
  
Install dependent packages from the environment.yml   
also: install conda and the nvidia drivers
```bash
sudo apt update
sudo apt install -y ubuntu-drivers-common
sudo ubuntu-drivers autoinstall
sudo reboot
#after reboot, check that nvidia tools are working
nvidia-smi
```

### Data Initialization

All local testing work was done by extracting subsets of sequences from s3 using boto3 as described above. 

## Processing Steps
Local testing -> Benchmarking on EC2 -> All(?) petadex sequences  
Sitetack provides all of their tools as individual .h5 files for each PTM, so helper functions are needed to load and use those models. The workflow was built using Tensorflow which is what the original authors used. These functions are defined below, but are also linked as separate .py files to compartmentalize the workflow. 

---

### Small-scale batch testing on all prediction models  
Functions compartmentalized to different files for organization purposes. 
`sitetack_general_functions.py` contains general helper functions needed for encoding sequences into tensors. 
`sitetack_model_functions_new.py` contains script for loading .h5 files, calls functions from `sitetack_general_functions.py`
For all models, a generic function was defined with the specific alphabets, residues ("PR" and "PR2"), and path as inputs for the functions. Partial functions defined for each model manually. Defined `predict_functions` as list of all partial functions. Also contains function for chunking large batches of sequences to prevent hitting GPU limits. 

In [ ]:
#define helper functions, no need to touch this
#import all of the required tools
import numpy as np
import tensorflow as tf
import pandas as pd
from tqdm import tqdm
from tensorflow import keras

def convert_to_onehot(data, alphabet):
    char_to_int = dict((c, i) for i, c in enumerate(alphabet))
    return [char_to_int[char] for char in data]

def tensor_encoding(x_data, depth, type, alphabet, k=53):
    indices = []
    for i in range(len(x_data)):
        indices.append(convert_to_onehot(x_data[i], alphabet))
        if len(convert_to_onehot(x_data[i], alphabet)) != k:
            print(x_data[i], "Length off")
    array = np.stack(indices, axis=0)
    if type == 'emb':
        return array
    t2 = []
    for i in tqdm(range(len(indices))):
        t1 = tf.one_hot(indices[i], depth)
        t2.append(t1)
    return t2

def get_kmer(seq, location, k=53):
    half = int((k - 1) / 2)
    if location > len(seq):
        print(f"Site outside of seq bounds, site: {location}, sequence length: {len(seq)}")
        return ''
    elif location <= half:
        if location > len(seq) - half:
            gap = "-" * (half - location + 1)
            gap2 = "-" * int(half - (len(seq) - location))
            kmer = seq[0:int(location + half)]
            kmer = gap + kmer + gap2
        else:
            gap = "-" * (half - location + 1)
            kmer = seq[0:int(location + half)]
            kmer = gap + kmer
    elif location > len(seq) - half:
        gap = "-" * int(half - (len(seq) - location))
        kmer = seq[int(location - half - 1): len(seq)]
        kmer = kmer + gap
    else:
        kmer = seq[int(location - half - 1): int(location + half)]
    assert len(kmer) == k
    return kmer

In [ ]:
#batch prediction function
def predict_ptm_batch(predict_fn, records, k_default=53):
    kw = predict_fn.keywords

    labeled_model = kw['labeled_model']
    unlabeled_model = kw['unlabeled_model']
    alphabet_with_labels = kw['alphabet_with_labels']
    alphabet_without_labels = kw['alphabet_without_labels']
    model = kw['model']
    PR = kw['PR']
    PR2 = kw['PR2']
    k = kw.get('k', k_default)

    all_kmers = []
    all_sites = []
    all_names = []

    for record in records:
        sequence = str(record.seq)
        seq_name = re.search(r'\|([^|]+)\|', record.description).group(1)

        PR_sites = [i + 1 for i, char in enumerate(sequence) if char == PR]
        PR_kmers = [get_kmer(sequence, s, k=k) for s in PR_sites]

        if PR2 == '':
            PR2_sites, PR2_kmers = [], []
        else:
            PR2_sites = [i + 1 for i, char in enumerate(sequence) if char == PR2]
            PR2_kmers = [get_kmer(sequence, s, k=k) for s in PR2_sites]

        kmers = PR_kmers + PR2_kmers
        sites = PR_sites + PR2_sites

        all_kmers.extend(kmers)
        all_sites.extend(sites)
        all_names.extend([seq_name] * len(kmers))

    # one batched encode + one batched predict call for ALL sequences
    tensor1 = tensor_encoding(all_kmers, 23, 'emb', alphabet_without_labels, k=k)
    tensor2 = tensor_encoding(all_kmers, 23, 'emb', alphabet_with_labels, k=k)

    nl_y_pred = unlabeled_model.predict(tensor1)[:, 0]
    l_y_pred = labeled_model.predict(tensor2)[:, 0]

    df = pd.DataFrame({
        "Name": all_names,
        "PTM Type": model,
        "Site": all_sites,
        "No labels model": nl_y_pred,
        "With PTM labels model": l_y_pred,
    })
    return df



I defined partial functions based off the parent function for each PTM type, if the paths are different depending on where the model is stored it will need to be changed. 

In [ ]:
#partial functions for each PTM type with specific alphabets, PR and PR2, paths
predict_n_glycosylation = partial(predict_ptm_batch, model = f"N-glycosylation(N)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@-UXZB", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UXZB", PR = "N", PR2 = "", labeled_model = tf.keras.models.load_model(f"./sitetack/models/N-Linked glycosylation (N)/All organism/N-linked_glycosylation_N_emb_CNN_with_labels_4.h5"), unlabeled_model = tf.keras.models.load_model(f"./sitetack/models/N-Linked glycosylation (N)/All organism/N-linked_glycosylation_N_emb_CNN_no_labels_4.h5"))
predict_ubiquitination = partial(predict_ptm_batch, model = f"Ubiquitination(K)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@-UX", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UX", PR = "K", PR2 = "", labeled_model = tf.keras.models.load_model(f"./sitetack/models/Ubiquitination (K)/All organism/Ubiquitin_K_emb_CNN_with_labels_7.h5"), unlabeled_model= tf.keras.models.load_model(f"./sitetack/models/Ubiquitination (K)/All organism/Ubiquitin_K_emb_CNN_no_labels_6.h5"))
predict_o_glycosylation = partial(predict_ptm_batch, model = f"O-glycosylation(S,T)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@&-UX", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UX", PR = "S", PR2 = "T", labeled_model = tf.keras.models.load_model(f"./sitetack/models/O-Linked glycosylation (S,T)/All organism/O-linked glycosylation_S_T_emb_CNN_with_labels_10.h5"), unlabeled_model = tf.keras.models.load_model(f"./sitetack/models/O-Linked glycosylation (S,T)/All organism/O-linked glycosylation_S_T_emb_CNN_no_labels_6.h5"))
predict_phosphorylation_st = partial(predict_ptm_batch, model = f"Phosphorylation(S,T)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@&-UXBZ", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UXBZ", PR = "S", PR2 = "T", labeled_model = tf.keras.models.load_model(f"./sitetack/models/Phosphorylation (S,T)/All organism/Phosphorylation_S_T_emb_CNN_with_labels_2.h5"), unlabeled_model = tf.keras.models.load_model(f"./sitetack/models/Phosphorylation (S,T)/All organism/Phosphorylation_S_T_emb_CNN_no_labels_6.h5"))
predict_phosphorylation_y = partial(predict_ptm_batch, model = f"Phosphorylation(Y)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@-UXBZ", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UXBZ", PR = "Y", PR2 = "", labeled_model = tf.keras.models.load_model(f"./sitetack/models/Phosphorylation (Y)/All organism/Phosphorylation_Y_emb_CNN_with_labels_8.h5"), unlabeled_model = tf.keras.models.load_model(f"./sitetack/models/Phosphorylation (Y)/All organism/Phosphorylation_Y_emb_CNN_no_labels_2.h5"))
predict_sumoylation = partial(predict_ptm_batch, model = f"SUMOylation(K)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@-UX", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UX", PR = "K", PR2 = "", labeled_model = tf.keras.models.load_model(f"./sitetack/models/SUMOylation (K)/All organism/SUMOylation_K_emb_CNN_with_labels_8.h5"), unlabeled_model = tf.keras.models.load_model(f"./sitetack/models/SUMOylation (K)/All organism/SUMOylation_K_emb_CNN_no_labels_8.h5"))
predict_acetylation = partial(predict_ptm_batch, model = f"Acetylation(K)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@-UX", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UX", PR = "K", PR2 = "", labeled_model = tf.keras.models.load_model(f"./sitetack/models/Acetylation (K)/All organism/N6-acetyllysine_K_emb_CNN_with_labels_1.h5"), unlabeled_model = tf.keras.models.load_model(f"./sitetack/models/Acetylation (K)/All organism/N6-acetyllysine_K_emb_CNN_no_labels_10.h5"))
predict_methylation_r = partial(predict_ptm_batch, model = f"Methylation(R)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@-UX", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UX", PR = "R", PR2 = "", labeled_model = tf.keras.models.load_model(f"./sitetack/models/Methylation (R)/All organism/Methylation_R_emb_CNN_with_labels_6.h5"), unlabeled_model = tf.keras.models.load_model(f"./sitetack/models/Methylation (R)/All organism/Methylation_R_emb_CNN_no_labels_6.h5"))
predict_methylation_k = partial(predict_ptm_batch, model = f"Methylation(K)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@-UX", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UX", PR = "K", PR2 = "", labeled_model = tf.keras.models.load_model(f"./sitetack/models/Methylation (K)/All organism/Methylation_K_emb_CNN_with_labels_5.h5"), unlabeled_model = tf.keras.models.load_model(f"./sitetack/models/Methylation (K)/All organism/Methylation_K_emb_CNN_no_labels_1.h5"))
predict_pyroglutamylation = partial(predict_ptm_batch, model = f"Pyroglutamylation(Q)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@-UXBZ", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UXBZ", PR = "Q", PR2 = "", labeled_model = tf.keras.models.load_model(f"./sitetack/models/Pyroglutamylation (Q)/All organism/Pyrrolidone-carboxylic-acid_Q_emb_CNN_with_labels_3.h5"), unlabeled_model = tf.keras.models.load_model(f"./sitetack/models/Pyroglutamylation (Q)/All organism/Pyrrolidone-carboxylic-acid_Q_emb_CNN_no_labels_1.h5"))
predict_palmitoylation = partial(predict_ptm_batch, model = f"Palmitoylation(C)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@-UX", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UX", PR = "C", PR2 = "", labeled_model = tf.keras.models.load_model(f"./sitetack/models/Palmitoylation (C)/All organism/S-Palmitoylation_C_emb_CNN_with_labels_4.h5"), unlabeled_model = tf.keras.models.load_model(f"./sitetack/models/Palmitoylation (C)/All organism/S-Palmitoylation_C_emb_CNN_no_labels_9.h5"))
predict_hydroxylation_p = partial(predict_ptm_batch, model = f"Hydroxylation(P)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@-UXZB", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UXZB", PR = "P", PR2 = "", labeled_model = tf.keras.models.load_model(f"./sitetack/models/Hydroxylation (P)/All organism/Hydroxyproline_P_emb_CNN_with_labels_1.h5"), unlabeled_model = tf.keras.models.load_model(f"./sitetack/models/Hydroxylation (P)/All organism/Hydroxyproline_P_emb_CNN_no_labels_7.h5"))
predict_hydroxylation_k = partial(predict_ptm_batch, model = f"Hydroxylation(K)", alphabet_with_labels = f"ARNDCEQGHILKMFPSTWYV@-UZ", alphabet_without_labels = f"ARNDCEQGHILKMFPSTWYV-UZ", PR = "K", PR2 = "", labeled_model = tf.keras.models.load_model(f"./sitetack/models/Hydroxylation (K)/All organism/Hydroxyproline_K_emb_CNN_with_labels_5.h5"), unlabeled_model = tf.keras.models.load_model(f"./sitetack/models/Hydroxylation (K)/All organism/Hydroxyproline_K_emb_CNN_no_labels_2.h5"))

predict_functions = [
    predict_n_glycosylation, predict_ubiquitination, predict_o_glycosylation,
    predict_phosphorylation_st, predict_phosphorylation_y, predict_sumoylation,
    predict_acetylation, predict_methylation_r, predict_methylation_k,
    predict_pyroglutamylation, predict_palmitoylation,
    predict_hydroxylation_p, predict_hydroxylation_k,
]

#chunking to prevent breaking due to GPU limits
def predict_ptm_batch_chunk(predict_fn, records, k_default=53, chunk_size=1000):
    all_dfs = []
    for i in range(0, len(records), chunk_size):
        chunk = records[i:i + chunk_size]
        df = predict_ptm_batch(predict_fn, chunk, k_default=k_default)
        all_dfs.append(df)
    return pd.concat(all_dfs, ignore_index=True)

This block runs everything together, in the file it will call all the functions from the other files. 

In [ ]:
#Batch predict script, run this
records = SeqIO.parse('blastnr_pazy.catalytic_orfs.fa', 'fasta')
first_10000 = list(islice(records, 10000))

#to prevent breaking for bad sequences containing non amino acid letters
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

def has_only_standard_aa(seq):
    return set(str(seq.seq)).issubset(STANDARD_AA)

first_10000 = [r for r in first_10000 if has_only_standard_aa(r)]

results_list = [predict_ptm_batch_chunk(fn, first_10000) for fn in predict_functions]
predict_results = pd.concat(results_list, ignore_index=True)
predict_results.to_csv('prediction_results_batch_gpu_10000.csv')

While this code is running you can monitor GPU usage with `watch -n 1 nvidia-smi` in a separate window to make sure tensorflow is using GPU. 

## Results & Discussion
I uploaded predictions for some 10,000 sequences from benchmarking to the s3. Most of the preds are junk, they need to be filtered for preds with score >0.8 or 0.9. 